# Streamlined MIMIC Experiment Results

Thin notebook for running or viewing the streamlined manuscript experiment artifacts.

In [1]:
# PROFILE selects the YAML config under manuscript/experiments/configs/.
# Options: "smoke" runs a small end-to-end experiment; "full" runs the manuscript-scale grid.
PROFILE = "smoke"

# RUN_EXPERIMENT controls whether this notebook executes experiments or only reloads saved artifacts.
# True: run the selected profile and overwrite generated artifact tables.
# False: view existing CSV artifacts from ARTIFACT_DIR without fitting models or fetching data.
RUN_EXPERIMENT = True

# RESTART controls resumability when RUN_EXPERIMENT=True.
# False: resume from existing raw/condition_results.csv and skip completed conditions.
# True: delete existing raw results first and rerun the selected profile from scratch.
RESTART = True

# SHOW_PROGRESS displays an updating text progress bar while RUN_EXPERIMENT=True.
# It is useful for smoke/full runs; set False if you prefer quieter notebook output.
SHOW_PROGRESS = True

# CONFIG_PATH can point to a custom YAML file. Leave as None to use configs/{PROFILE}.yaml.
CONFIG_PATH = None

# ARTIFACT_DIR controls where raw results, tables, figures, and reports are written/read.
# Leave as None to use manuscript/experiments/artifacts/.
ARTIFACT_DIR = None

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from pathlib import Path
import sys
import pandas as pd

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 240)

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "MIMIC" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

EXPERIMENT_ROOT = PROJECT_ROOT / "manuscript" / "experiments"
for path in [PROJECT_ROOT / "src", EXPERIMENT_ROOT / "src"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print(f"Python executable: {sys.executable}")
try:
    import scikit_posthocs
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "scikit-posthocs is required for critical difference diagrams. "
        f"Install it in this notebook kernel with: {sys.executable} -m pip install scikit-posthocs"
    ) from exc

from streamlined.config import artifact_paths, ensure_artifact_dirs, load_config
from streamlined.datasets import dataset_registry
from streamlined.runner import artifact_manifest, run_profile
from streamlined.plotting import generate_critical_difference_diagram, generate_mean_learning_curves, plot_rank_summary

Python executable: /home/fabrizio/.venvs/py314/bin/python


In [4]:
config_path = Path(CONFIG_PATH) if CONFIG_PATH else EXPERIMENT_ROOT / "configs" / f"{PROFILE}.yaml"
artifact_dir = ARTIFACT_DIR or str(EXPERIMENT_ROOT / "artifacts")
config = load_config(config_path, artifact_dir=artifact_dir)
ensure_artifact_dirs(config)
print(f"Artifact directory: {config.artifact_root}")
display(dataset_registry())
manifest = artifact_manifest(config)
display(manifest[["artifact", "relative_path", "exists"]])

Artifact directory: /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/MIMIC/manuscript/experiments/artifacts


,key,name,openml_id,target,n_rows,numeric_features,categorical_features,status
0,adult,Adult,1590,class,48842,6,8,ready
1,bank_marketing,Bank Marketing,44234,y,45211,7,9,ready
2,default_credit,Default of Credit Card Clients,42477,y,30000,14,9,ready


,artifact,relative_path,exists
0,raw_results,raw/condition_results.csv,True
1,learning_curves,tables/learning_curves.csv,True
2,aulc,tables/aulc.csv,True
3,pairwise,tables/pairwise_comparisons.csv,True
4,regime,tables/regime_summary.csv,True
5,rank,tables/rank_summary.csv,True
6,conclusions,reports/prescriptive_conclusions.md,True
7,figures,figures,True


In [ ]:
tables = run_profile(config, run_experiment=RUN_EXPERIMENT, show_progress=SHOW_PROGRESS, restart=RESTART)
display(tables["raw_results"].head())
display(tables["learning_curves"].head())
display(tables["aulc"].head())
display(tables["pairwise"].head())
display(tables["regime"].head())

[##----------------------------] 10/120 elapsed=3m18s avg=20s/step ETA=36m15s adult ratio=2:1 n=256 seed=0 method=latent_displacement


In [ ]:
if not tables["learning_curves"].empty:
    print('Higher ROC AUC is better')
    ratio = sorted(tables["learning_curves"]["imbalance_ratio"].unique())[0]
    fig, ax = generate_mean_learning_curves(tables["learning_curves"], imbalance_ratio=ratio)
    display(fig)
if not tables["aulc"].empty:
    print('Lower rank is better')
    for segment in ["full", "early"]:
        fig, ax = generate_critical_difference_diagram(tables["aulc"], segment=segment)
        display(fig)
